In [7]:
# imports 

import subprocess
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time
import requests
import re
import csv

In [8]:
# # run at start of session only

# # Check versions
# result = subprocess.run(
#     ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome", "--version"],
#     capture_output=True, text=True
# )
# print(result.stdout)

# print("Selenium version:", selenium.__version__)

# options = Options()
# options.add_argument("--headless")
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")
# options.add_argument("--window-size=1920,1080")

# driver = webdriver.Chrome(options=options)  # No Service() or ChromeDriverManager
# driver.get("https://www.google.com")
# print("Success! Title:", driver.title)
# driver.quit()

In [9]:
# Returns imdb reviews url for a given movie name with spaces, e.g. "The Batman"
def get_imdb_reviews_url(driver, movie_name):
    search_query = movie_name.replace(" ", "+")
    search_url = f"https://www.imdb.com/find/?q={search_query}&s=tt&ttype=ft"

    driver.get(search_url)
    time.sleep(5)  # Let the page fully load
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    first_result = soup.find("a", attrs={"class": "ipc-title-link-wrapper"})  # Updated class
    
    if not first_result:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    href = first_result["href"]
    movie_id = href.split("/")[2]
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    return movie_url, reviews_url


# Scrapes movie information including movie title, iMDB rating, year, budget, and gross income across US and Canada
def get_movie_info_imdb(driver, main_url):
    print("Scraping movie info...")
    driver.get(main_url)
    time.sleep(5)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    info = {}

    # Title
    tag = soup.find(attrs={"data-testid": "hero__primary-text"})
    info["title"] = tag.get_text(strip=True) if tag else "N/A"

    # Rating
    tag = soup.find(attrs={"data-testid": "hero-rating-bar__aggregate-rating__score"})
    info["imdb_rating"] = tag.get_text(strip=True).replace("/10", "") if tag else "N/A"

    # Year
    tag = soup.find(attrs={"data-testid": "hero-parent"})
    if tag:
        match = re.search(r'\b(19|20)\d{2}\b', tag.get_text())
        info["year"] = match.group(0) if match else "N/A"
    else:
        info["year"] = "N/A"

    # Budget
    tag = soup.find(attrs={"data-testid": "title-boxoffice-budget"})
    if tag:
        text = tag.get_text(strip=True).replace("Budget", "").replace("(estimated)", "").strip()
        info["budget"] = text
    else:
        info["budget"] = "N/A"

    # Gross US & Canada
    tag = soup.find(attrs={"data-testid": "title-boxoffice-grossdomestic"})
    if tag:
        text = tag.get_text(strip=True).replace("Gross US & Canada", "").strip()
        info["gross_us"] = text
    else:
        info["gross_us"] = "N/A"

    print(f"Movie info: {info}")
    return info



# Scrape a movie's top 50 featured reviews
def scrape_reviews_imdb(driver, reviews_url):
    print("Scraping reviews...")
    driver.get(reviews_url)
    time.sleep(5)

    # Click "Load More" twice to scrape 75 reviews in total then truncate to 50
    for i in range(2):
        try:
            load_more = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more)
            time.sleep(3)
            print("Loaded more reviews...")
        except:
            print("All reviews loaded.")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    reviews = []
    articles = soup.find_all("article", class_="user-review-item")
    print(f"Found {len(articles)} reviews, parsing...")

    count = 1

    for article in articles:
        review = {}

        review["id"] = f"{count}"
        count += 1

        # Rating — only present if user gave one
        rating_tag = article.find("span", class_="ipc-rating-star--rating")
        review["user_rating"] = rating_tag.get_text(strip=True) if rating_tag else "N/A"

        # Title
        title_tag = article.find("h3", class_="ipc-title__text")
        review["review_title"] = title_tag.get_text(strip=True) if title_tag else "N/A"

        # Review text
        text_tag = article.find("div", attrs={"data-testid": "review-overflow"})
        if not text_tag:
            text_tag = article.find("div", class_=lambda c: c and "content" in c.lower())
        review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

        # Author
        author_tag = article.find("a", attrs={"data-testid": "author-link"})
        review["author"] = author_tag.get_text(strip=True) if author_tag else "N/A"

        # Date
        date_tag = article.find("li", attrs={"data-testid": "review-date"})
        review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

        reviews.append(review)

    reviews = reviews[:50]
    return reviews


In [10]:
def get_imdb_reviews_url_api(movie_name):
    # Use IMDB's suggestion API — fast, no Selenium needed
    search_query = movie_name.replace(" ", "_")
    api_url = f"https://v3.sg.media-imdb.com/suggestion/x/{search_query}.json"
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    response = requests.get(api_url, headers=headers)
    data = response.json()
    
    # Filter for movies only (qid == "movie") and grab the first one
    movies = [r for r in data.get("d", []) if r.get("qid") == "movie"]
    
    if not movies:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    movie_id = movies[0]["id"]  # e.g. tt6751668
    title = movies[0]["l"]

    # clean title for use in naming reviews csv
    title_clean = re.sub(r"[^\w]", "_", title.lower())
    title_clean = re.sub(r"_+", "_", title_clean).strip("_") 
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    print(f"Found: {title} → {movie_url}")
    print(f"Found: {title} Reviews → {reviews_url}")
    return movie_url, reviews_url, title_clean

### main

In [13]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

movie_url, reviews_url, title_clean = get_imdb_reviews_url_api("Parasite")
movie_info = get_movie_info_imdb(driver, movie_url)
reviews = scrape_reviews_imdb(driver, reviews_url)

driver.quit()

print(f"\nScraped {len(reviews)} reviews total.")

# Save to CSV
filepath = f'/Users/Diane/Desktop/PSYCH 186B/project/reviews/{title_clean}_imdb_reviews.csv'
with open(filepath, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["id", "author", "date", "user_rating", "review_title", "review_text"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(reviews)

print(f"Saved to: {filepath}")

Found: Parasite → https://www.imdb.com/title/tt6751668/
Found: Parasite Reviews → https://www.imdb.com/title/tt6751668/reviews/
Scraping movie info...
Movie info: {'title': 'Parasite', 'imdb_rating': '8.5', 'year': 'N/A', 'budget': '$11,400,000', 'gross_us': '$53,847,897'}
Scraping reviews...
Loaded more reviews...
Loaded more reviews...
Found 74 reviews, parsing...

Scraped 50 reviews total.
Saved to: /Users/Diane/Desktop/PSYCH 186B/project/reviews/parasite_imdb_reviews.csv
